# II. Bivariate Analysis

This notebook explores relationships between key marketing variables to uncover business insights.

Analyses included:
- Channel efficiency using ROI distributions by channel
- Audience targeting performance using average conversion rate
- Correlation structure across numerical campaign metrics
- Statistical significance test for ROI differences between Google Ads and YouTube

Assumption: `df` is already loaded from the cleaned marketing dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

## Step 1: Data Validation for Required Columns

Before analysis, validate that the dataset includes the columns needed for plotting and statistical testing.

In [ ]:
required_columns = [
    "Channel_Used",
    "ROI",
    "Target_Audience",
    "Conversion_Rate",
    "Acquisition_Cost",
    "Clicks",
    "Impressions",
    "Engagement_Score",
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("All required columns are available.")

## Step 2: Channel Efficiency

Compare ROI distributions across channels to evaluate which channel tends to deliver stronger returns.

The boxplot highlights medians, spread, and potential outliers by channel.

In [ ]:
plot_df = df[["Channel_Used", "ROI"]].dropna().copy()

plt.figure(figsize=(12, 6))
sns.boxplot(data=plot_df, x="Channel_Used", y="ROI", palette="Set2")
plt.title("ROI Distribution by Marketing Channel")
plt.xlabel("Channel Used")
plt.ylabel("ROI")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

channel_roi_summary = (
    plot_df.groupby("Channel_Used")["ROI"]
    .agg(["count", "mean", "median", "std"])
    .sort_values(by="median", ascending=False)
)
channel_roi_summary

### Channel Efficiency Interpretation

The most cost-effective channel should be interpreted as the one with:
- The highest **median ROI** (more robust than mean against outliers), and
- A relatively stable spread (lower variability where possible).

Use the table shown above (`channel_roi_summary`) to identify the top channel after execution.

## Step 3: Audience Targeting

Compare average conversion rates by target audience to understand which audience responds best to campaigns.

In [ ]:
audience_df = df[["Target_Audience", "Conversion_Rate"]].dropna().copy()

audience_order = (
    audience_df.groupby("Target_Audience")["Conversion_Rate"]
    .mean()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=audience_df,
    x="Target_Audience",
    y="Conversion_Rate",
    order=audience_order,
    estimator=np.mean,
    errorbar="sd",
    palette="Blues_d"
)
plt.title("Average Conversion Rate by Target Audience")
plt.xlabel("Target Audience")
plt.ylabel("Average Conversion Rate")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## Step 4: Correlation Analysis

Evaluate linear relationships among key numerical metrics to detect trade-offs and synergies across campaign performance indicators.

In [ ]:
num_cols = [
    "Conversion_Rate",
    "Acquisition_Cost",
    "ROI",
    "Clicks",
    "Impressions",
    "Engagement_Score",
]

corr_df = df[num_cols].copy()
corr_matrix = corr_df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot_kws={"size": 10}
)
plt.title("Correlation Heatmap: Marketing Performance Metrics")
plt.tight_layout()
plt.show()

## Step 5: Statistical Significance: Google Ads vs YouTube ROI

Use a two-sample Welch's t-test to compare average ROI between **Google Ads** and **YouTube**.

- Null hypothesis (`H0`): the mean ROI is equal for both channels.
- Alternative hypothesis (`H1`): the mean ROI differs between channels.

In [ ]:
alpha = 0.05

google_roi = df.loc[df["Channel_Used"] == "Google Ads", "ROI"].dropna()
youtube_roi = df.loc[df["Channel_Used"] == "YouTube", "ROI"].dropna()

if len(google_roi) < 2 or len(youtube_roi) < 2:
    raise ValueError(
        f"Insufficient data for t-test. Google Ads n={len(google_roi)}, YouTube n={len(youtube_roi)}"
    )

t_stat, p_value = stats.ttest_ind(google_roi, youtube_roi, equal_var=False)

print(f"Google Ads sample size: {len(google_roi)}")
print(f"YouTube sample size: {len(youtube_roi)}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < alpha:
    print(
        f"Result: Statistically significant difference in ROI between Google Ads and YouTube (p < {alpha})."
    )
else:
    print(
        f"Result: No statistically significant ROI difference between Google Ads and YouTube (p >= {alpha})."
    )

## Business Takeaways

Use the outputs above to summarize:
- Which channels show stronger ROI efficiency and stability,
- Which audience segments convert best on average,
- Which metric pairs move together strongly, and
- Whether Google Ads vs YouTube ROI differences are statistically meaningful.

These insights can directly guide channel allocation and audience targeting strategy.